# AFib Detection with CNN-BiLSTM

This notebook implements the research/educational Normal-vs-AFib ECG classification pipeline. It uses the PhysioNet/CinC 2017 training data and does not include final performance claims until a reproducible run is completed.

In [ ]:
# Colab/runtime dependencies
!wget -q https://physionet.org/files/challenge-2017/1.0.0/training2017.zip
!unzip -q training2017.zip


## Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import scipy.io as sio
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
)

DATA_PATH = "./training2017"
SEQ_LEN = 2700
FS = 300
RANDOM_STATE = 42


## 1. Data loading

In [ ]:
def load_physionet_cinc2017(data_path, seq_len=2700):
    print("Gerçek PhysioNet verileri yükleniyor...")
    csv_path = os.path.join(data_path, "REFERENCE.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"{csv_path} bulunamadı! Lütfen veri setini indirdiğinizden emin olun.")

    ref_df = pd.read_csv(csv_path, header=None, names=["record", "label"])
    ref_df = ref_df[ref_df["label"].isin(["N", "A"])]

    X, y = [], []
    for _, row in ref_df.iterrows():
        record_name = row["record"]
        label = row["label"]
        mat_file = os.path.join(data_path, f"{record_name}.mat")
        if not os.path.exists(mat_file):
            continue

        mat_data = sio.loadmat(mat_file)
        ecg_signal = mat_data["val"][0]
        if len(ecg_signal) >= seq_len:
            ecg_signal = ecg_signal[:seq_len]
        else:
            ecg_signal = np.pad(ecg_signal, (0, seq_len - len(ecg_signal)), mode="constant")

        X.append(ecg_signal)
        y.append(0 if label == "N" else 1)

    print(f"Toplam {len(X)} adet gerçek EKG kaydı başarıyla yüklendi.")
    return np.asarray(X), np.asarray(y)

X_raw, y = load_physionet_cinc2017(DATA_PATH, seq_len=SEQ_LEN)


## 2. Signal preprocessing

In [ ]:
def apply_filters(X, fs=300):
    print("Sinyal İşleme: Bandpass (0.5-45 Hz) ve Notch (50 Hz) filtreleri uygulanıyor...")
    X_filtered = np.zeros_like(X, dtype=np.float64)
    b_notch, a_notch = signal.iirnotch(w0=50.0, Q=30.0, fs=fs)
    nyq = 0.5 * fs
    low = 0.5 / nyq
    high = 45.0 / nyq
    b_band, a_band = signal.butter(N=4, Wn=[low, high], btype="band")

    for i in range(X.shape[0]):
        temp_sig = signal.filtfilt(b_notch, a_notch, X[i])
        X_filtered[i] = signal.filtfilt(b_band, a_band, temp_sig)
    return X_filtered

def apply_zscore_normalization(X):
    print("Veriler Z-Score ile normalize ediliyor...")
    mean = np.mean(X, axis=1, keepdims=True)
    std = np.std(X, axis=1, keepdims=True)
    return (X - mean) / (std + 1e-8)

X_filt = apply_filters(X_raw, fs=FS)
X_norm = apply_zscore_normalization(X_filt)
X_final = X_norm.reshape((X_norm.shape[0], SEQ_LEN, 1))

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Eğitim seti boyutu: {X_train.shape} | Test seti boyutu: {X_test.shape}")


## 3. CNN-BiLSTM model

In [ ]:
def build_hybrid_model(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(Conv1D(filters=64, kernel_size=15, activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=128, kernel_size=10, activation="relu", padding="same", name="last_conv"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    model.add(Bidirectional(LSTM(64, return_sequences=False)))
    model.add(Dropout(0.4))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
    )
    return model

model = build_hybrid_model((SEQ_LEN, 1))
model.summary()


## 4. Training

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=0.00001)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)


## 5. Evaluation

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=["Normal", "AFib"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "AFib"], yticklabels=["Normal", "AFib"])
plt.title("Confusion Matrix (Real ECG Data)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


## 6. 1-D Grad-CAM

In [ ]:
def make_gradcam_heatmap_1d(img_array, model, last_conv_layer_name="last_conv"):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_output, preds = grad_model(img_array)
        class_channel = preds[:, 0]  # sigmoid AFib output

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = tf.math.divide_no_nan(heatmap, tf.reduce_max(heatmap))
    return heatmap.numpy()

afib_indices = np.where((y_test == 1) & (y_pred == 1))[0]
if len(afib_indices) > 0:
    test_idx = afib_indices[0]
    sample_signal = X_test[test_idx:test_idx+1]
    heatmap = make_gradcam_heatmap_1d(sample_signal, model)
    heatmap_resized = np.interp(np.linspace(0, 1, SEQ_LEN), np.linspace(0, 1, len(heatmap)), heatmap)

    plt.figure(figsize=(12, 4))
    plt.plot(sample_signal[0].flatten(), alpha=0.7, label="Z-score ECG signal")
    plt.imshow(
        heatmap_resized[np.newaxis, :], cmap="jet", aspect="auto", alpha=0.4,
        extent=[0, SEQ_LEN, np.min(sample_signal), np.max(sample_signal)]
    )
    plt.colorbar(label="AI attention intensity")
    plt.title("1-D Grad-CAM for an AFib Prediction")
    plt.xlabel("Time step")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.tight_layout()
    plt.show()
